In [24]:
!pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.51  Python-3.13.5 torch-2.12.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
Setup complete  (8 CPUs, 7.7 GB RAM, 214.3/237.4 GB disk)


In [1]:
!yolo predict model=yolo26n.pt source='images/'

Ultralytics 8.4.51  Python-3.13.5 torch-2.12.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

image 1/1565 C:\Users\keert\images\00000000.jpg: 448x640 1 car, 177.7ms
image 2/1565 C:\Users\keert\images\00000001.jpg: 448x640 1 train, 156.5ms
image 3/1565 C:\Users\keert\images\00000002.jpg: 448x640 (no detections), 96.3ms
image 4/1565 C:\Users\keert\images\00000003.jpg: 448x640 (no detections), 100.3ms
image 5/1565 C:\Users\keert\images\00000004.jpg: 448x640 1 person, 1 boat, 84.3ms
image 6/1565 C:\Users\keert\images\00000005.jpg: 448x640 2 trucks, 110.7ms
image 7/1565 C:\Users\keert\images\00000006.jpg: 448x640 (no detections), 108.9ms
image 8/1565 C:\Users\keert\images\00000007.jpg: 448x640 (no detections), 177.4ms
image 9/1565 C:\Users\keert\images\00000008.jpg: 448x640 1 truck, 118.2ms
image 10/1565 C:\Users\keert\images\00000009.jpg: 448x640 3 cars, 112.8ms
image 11/1565 C:\Users\keert\images\00000

In [18]:
import pandas as pd
import os

# Load your CSV
labels_df = pd.read_csv("labels.csv")
labels_df.columns = ['image_id','class','xmin','ymin','xmax','ymax']
labels_df['image_id'] = labels_df['image_id'].apply(lambda x: f"{x:08d}")
labels_df =labels_df.iloc[:1000]

In [19]:
# Get unique class names
class_names = sorted(labels_df['class'].unique().tolist())

# Build mapping: class name → numeric ID
class_map = {name: idx for idx, name in enumerate(class_names)}
print(class_map)

{'articulated_truck': 0, 'bicycle': 1, 'bus': 2, 'car': 3, 'motorcycle': 4, 'motorized_vehicle': 5, 'non-motorized_vehicle': 6, 'pedestrian': 7, 'pickup_truck': 8, 'single_unit_truck': 9, 'work_van': 10}


In [20]:
import shutil
import time
# Example columns in CSV: image_id, class, xmin, ymin, xmax, ymax
# Adjust names if your CSV differs
images_dir = "images" 
labels_dir = "labels" 
# Delete the folder if it exists
if os.path.exists(labels_dir):
    shutil.rmtree(labels_dir)# folder to save YOLO txt files
os.makedirs(labels_dir, exist_ok=True)

for index, row in labels_df.iterrows():
    image_id = row['image_id']
    class_name = row['class'].strip().lower()   # removes spaces, converts to lowercase
    class_id = class_map[class_name]

    # bounding box coordinates
    xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']

    # get image size (needed for normalization)
    import cv2
    img = cv2.imread(os.path.join(images_dir, f"{image_id}.jpg"))
    h, w = img.shape[:2]

    # convert to YOLO format
    x_center = ((xmin + xmax) / 2) / w
    y_center = ((ymin + ymax) / 2) / h
    width = (xmax - xmin) / w
    height = (ymax - ymin) / h
    
    # write to txt file
    label_path = os.path.join(labels_dir, f"{image_id}.txt")
    with open(label_path, "w") as f:   # "a" allows multiple boxes per image
        f.write(f"{class_id} {x_center} {y_center} {width} {height}\n")


In [21]:
import os, random, shutil

images_dir = "images"
labels_dir = "labels"

train_img_dir = "dataset/images/train"
val_img_dir   = "dataset/images/val"
train_lbl_dir = "dataset/labels/train"
val_lbl_dir   = "dataset/labels/val"

# --- CLEAN OLD DATASET ---
for d in ["dataset/images/train", "dataset/images/val",
          "dataset/labels/train", "dataset/labels/val"]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
                
# create folders
for d in [train_img_dir, val_img_dir, train_lbl_dir, val_lbl_dir]:
    os.makedirs(d, exist_ok=True)

# get all image files
image_files = [f for f in os.listdir(images_dir) if f.endswith(".jpg")][:300]
print(len(image_files))
random.shuffle(image_files)

# split 80/20
split_idx = int(0.8 * len(image_files))
train_files = image_files[:split_idx]
val_files   = image_files[split_idx:]

# move images + labels
for f in train_files:
    shutil.copy(os.path.join(images_dir, f), train_img_dir)
    shutil.copy(os.path.join(labels_dir, f.replace(".jpg", ".txt")), train_lbl_dir)

for f in val_files:
    shutil.copy(os.path.join(images_dir, f), val_img_dir)
    shutil.copy(os.path.join(labels_dir, f.replace(".jpg", ".txt")), val_lbl_dir)


300


In [22]:
data_yaml = f"""
train: dataset/images/train
val: dataset/images/val

nc: {len(class_names)}
names: {class_names}
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created successfully!")

data.yaml created successfully!


In [23]:
!yolo val model=yolo26n.pt data='data.yaml'

Ultralytics 8.4.51  Python-3.13.5 torch-2.12.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)

OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already initialized.
OMP: Hint This means that multiple copies of the OpenMP runtime have been linked into the program. That is dangerous, since it can degrade performance or cause incorrect results. The best thing to do is to ensure that only a single OpenMP runtime is linked into the process, e.g. by avoiding static linking of the OpenMP runtime in any library. As an unsafe, unsupported, undocumented workaround you can set the environment variable KMP_DUPLICATE_LIB_OK=TRUE to allow the program to continue to execute, but that may cause crashes or silently produce incorrect results. For more information, please see http://www.intel.com/software/products/support/.



YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 30.612.9 MB/s, size: 20.6 KB)

val: Scanning C:\Users\keert\dataset\labels\val... 60 images, 0 backgrounds, 0 corrupt: 100% â”�â”�â”�â”�â”�â”�â”�â”�â”�â”�â”�â”� 60/60 842.4it/s 0.1s
val: New cache created: C:\Users\keert\dataset\labels\val.cache

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 25% â”�â”�â”�â”€â”€â”€â”€â”€â”€â”€â”€â”€ 1/4 8.7s/it 2.6s<26.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 50% â”�â”�â”�â”�â”�â”�â”€â”€â”€â”€â”€â”€ 2/4 4.3s/it 4.6s<8.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 75% â”�â”�â”�â”�â”�â”�â”�â”�â”�â”€â”€â”€ 3/4 3.4s/it 6.8s<3.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% â”�â”�â”�â”�â”�â”�â”�â”�â”�â”�â”�â”� 4/4 2.1s/it 8.3s


In [27]:
!yolo mode=predict model=yolo26n.pt source="dataset/images/val" save=True data='data.yaml' 

Ultralytics 8.4.51  Python-3.13.5 torch-2.12.0+cpu CPU (11th Gen Intel Core i5-1135G7 @ 2.40GHz)
YOLO26n summary (fused): 122 layers, 2,408,932 parameters, 0 gradients, 5.4 GFLOPs

image 1/60 C:\Users\keert\dataset\images\val\00000001.jpg: 448x640 1 train, 194.4ms
image 2/60 C:\Users\keert\dataset\images\val\00000004.jpg: 448x640 1 person, 1 boat, 141.4ms
image 3/60 C:\Users\keert\dataset\images\val\00000012.jpg: 448x640 3 cars, 168.7ms
image 4/60 C:\Users\keert\dataset\images\val\00000013.jpg: 448x640 (no detections), 113.6ms
image 5/60 C:\Users\keert\dataset\images\val\00000020.jpg: 448x640 (no detections), 112.4ms
image 6/60 C:\Users\keert\dataset\images\val\00000031.jpg: 448x640 1 car, 105.2ms
image 7/60 C:\Users\keert\dataset\images\val\00000040.jpg: 448x640 3 cars, 93.5ms
image 8/60 C:\Users\keert\dataset\images\val\00000044.jpg: 448x640 1 car, 1 boat, 105.5ms
image 9/60 C:\Users\keert\dataset\images\val\00000046.jpg: 448x640 2 buss, 127.2ms
image 10/60 C:\Users\keert\dataset\ima